cleaning & feature engineering 

In [1]:
import pandas as pd
import numpy as np
import ast
import re

In [2]:
#loading the data
df = pd.read_csv('../Data/data.csv')

In [3]:
df_clean = df.copy()

In [4]:
# Drop dead columns
# We use errors='ignore' just in case you run the cell twice
df_clean = df_clean.drop(columns=['delivery_details', 'delivery_fee'], errors='ignore')

In [5]:
print(f"Original columns: {len(df.columns)}")
print(f"Cleaned columns: {len(df_clean.columns)}")
print("Dead columns dropped. Ready for Task 2 (Brand Recovery).")

Original columns: 22
Cleaned columns: 20
Dead columns dropped. Ready for Task 2 (Brand Recovery).


In [7]:
# Create a mask to see rows that originally had no brand 
# (Assuming 'brand' in raw data was NaN for these)
originally_missing = df['brand'].isna()

print(f"Total brands recovered: {originally_missing.sum()}")
print("-" * 50)

# Spot-check 20 random samples from the recovered group
check_df = df_clean[originally_missing][['title', 'brand']].sample(20, random_state=42)

# Styling the output for easier manual reading
display(check_df)

# Validation logic
remaining_nulls = df_clean['brand'].isnull().sum()
null_percentage = (remaining_nulls / len(df_clean)) * 100

print("-" * 50)
print(f"Current Brand Null Count: {remaining_nulls} ({null_percentage:.2f}%)")
if null_percentage < 5:
    print(" SUCCESS: Brand nulls are below the 5% target!")
else:
    print(" WARNING: Still above 5% nulls. Add more brands to KNOWN_BRANDS.")

Total brands recovered: 893
--------------------------------------------------


,title,brand
1483,XMobile Dabang Plus,NaN
1213,WS27 Bluetooth Calling Watch,NaN
1298,TG-38 Ultra Smartwatch,NaN
1494,Dcode Cygnal 2 Lite,NaN
812,Samsung Galaxy Buds Pro,NaN
1063,QCY T1C TWS Bluetooth Earphones,NaN
1073,Audionic Bluetooth Neckband (B730),NaN
1106,1More Omthing PistonBuds TWS Bluetooth Earbuds,NaN
981,Audionic Signature Premium Neckband (N220),NaN
909,HOTTU TWS Earpods (P73 Max),NaN


--------------------------------------------------
Current Brand Null Count: 893 (53.60%)


In [8]:
KNOWN_BRANDS = [
    # Multi-word brands first
    'Red Magic', 'VGO TEL', 'Dcode Cygnal', 'Club Mobile', 'Mobile Dabang', 
    '1More Omthing', '1More',
    # Single word brands
    'Samsung', 'Apple', 'Xiaomi', 'Oppo', 'Vivo', 'Realme', 'OnePlus', 
    'Nokia', 'Google', 'Infinix', 'Tecno', 'Huawei', 'Motorola', 'Lenovo', 
    'HP', 'Dell', 'Asus', 'Acer', 'MSI', 'Razer', 'Sony', 'JBL', 'Anker', 
    'QCY', 'Sennheiser', 'Bose', 'Nothing', 'Sparx', 'Hisense', 'Audionic', 
    'Ronin', 'Joyroom', 'Baseus', 'HOTTU', 'Calme', 'Ansty', 'WS27', 'Dcode'
]

In [9]:
def recover_brand(row):
    # Check if existing brand is valid
    existing = str(row['brand']).strip()
    if pd.notnull(row['brand']) and existing.lower() != 'nan' and existing != '':
        return row['brand']
    
    title = str(row['title'])
    
    # Logic A: Priority Scan for Known Brands
    for brand in KNOWN_BRANDS:
        if brand.lower() in title.lower():
            return brand
            
    # Logic B: Fallback - First word if it starts with a Capital
    match = re.search(r'\b[A-Z][a-zA-Z0-9]*\b', title)
    if match:
        return match.group(0)
    
    return "Unknown" # Never return actual NaN/Null here

# APPLY AND CHECK
df_clean['brand'] = df_clean.apply(recover_brand, axis=1)

# Verification
new_null_count = df_clean['brand'].isna().sum()
unknown_count = (df_clean['brand'] == 'Unknown').sum()
print(f"New Null Count: {new_null_count}")
print(f"Unknowns (Fallback failed): {unknown_count}")

New Null Count: 0
Unknowns (Fallback failed): 0
